# 🧠 Multi-Agent Computer Architecture Teaching Assistant

This notebook runs the **multi-agent RAG system** using LangChain + LangGraph.

**Agents:**
- 📚 **RAG Agent** — Textbook search & Q&A
- 🧮 **Math Agent** — Calculations, CPI, Amdahl's Law
- 📖 **Knowledge Agent** — Definitions, summaries, comparisons
- 💻 **Code Agent** — Assembly code tracing & generation

**Tools:** Textbook Search, Calculator, Glossary, Chapter Summarizer, Concept Comparison, Assembly Tracer, Web Search

In [15]:
# Cell 1: Mount Google Drive
from google.colab import drive
import os

if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# Clone or navigate to project
PROJECT_DIR = '/content/drive/My Drive/Colab_RAG_Project'
if os.path.exists(PROJECT_DIR):
    os.chdir(PROJECT_DIR)
    print(f'✅ Working directory: {os.getcwd()}')
else:
    print(f'⚠️ Project directory not found at {PROJECT_DIR}')
    print('Please update PROJECT_DIR to point to your project folder.')

✅ Working directory: /content/drive/My Drive/Colab_RAG_Project


In [16]:
# Cell 2: Install dependencies
!pip install -q langchain>=0.2.0 langchain-community>=0.2.0 langchain-huggingface>=0.0.3
!pip install -q langgraph>=0.1.0
!pip install -q transformers==4.43.3 accelerate sentence-transformers bitsandbytes
!pip install -q faiss-cpu --no-deps
!pip install -q gradio>=4.0.0 duckduckgo-search>=5.0.0 numexpr>=2.8.0
print('✅ All dependencies installed.')

✅ All dependencies installed.


In [17]:
# Cell 3: Initialize the multi-agent system
from agents.config import Config
from agents.graph import build_graph, run_query
from agents.memory import ConversationMemory

# Configure for Colab
# By default, we point 'chapters_db_path' to 'slides_db' to use the lecture slides RAG database.
# To switch back to textbook chapters, set chapters_db_path to '/content/drive/My Drive/Colab_RAG_Project/chapters_db'
config = Config(
    chapters_db_path='/content/drive/My Drive/Colab_RAG_Project/slides_db',
    model_name='microsoft/Phi-3.5-mini-instruct',
)
print(config.summary())

# Build the agent graph (loads LLM + initializes all tools)
graph = build_graph(config)
memory = ConversationMemory(max_turns=config.max_history)

print('\n\n✅ Multi-Agent System Ready!')

⚠️  Warning: chapters_db not found at /content/drive/My Drive/Colab_RAG_Project/slides_db
   Run separate_chps.py first, or set config.chapters_db_path manually.
╔══════════════════════════════════════════╗
║        Agent System Configuration        ║
╠══════════════════════════════════════════╣
║  Environment : colab                     ║
║  Backend     : huggingface               ║
║  Device      : cuda                      ║
║  LLM         : Phi-3.5-mini-instruct     ║
║  Embedder    : all-MiniLM-L6-v2          ║
║  Top-K       : 5                         ║
║  Memory      : 10                        turns ║
║  DB Path     : .../slides_db             ║
╚══════════════════════════════════════════╝
🔧 Building agent graph …
🔧 Initialising tools …
📚 RAG Tool: Loading embedder 'all-MiniLM-L6-v2' ...
  ⚠️  init_rag_tool() failed: [Errno 2] No such file or directory: '/content/drive/My Drive/Colab_RAG_Project/slides_db'
✅ Summarizer tool initialized
  ✅ init_summarizer_tool() initialised
✅ 

In [9]:
# Cell 4: Test the agents

test_queries = [
    'What is the difference between RISC and CISC?',    # -> RAG Agent
    'Calculate 2**10',                                    # -> Math Agent
    'Define pipeline hazard',                             # -> Knowledge Agent
    'Write code to add two numbers',                 # -> Code Agent
]

for query in test_queries:
    print(f'\n{"="*60}')
    print(f'📝 Query: {query}')
    print(f'{"="*60}')

    result = run_query(graph, query, memory)

    print(f'\n🤖 Agent: {result["agent_used"]}')
    print(f'\n{result["response"]}')

    if result.get('tool_calls_log'):
        print(f'\n🔧 Tools: {", ".join(result["tool_calls_log"])}')
    print()


📝 Query: What is the difference between RISC and CISC?
🔀 Supervisor routed to: KNOWLEDGE_AGENT


The `seen_tokens` attribute is deprecated and will be removed in v4.41. Use the `cache_position` model input instead.


📖 Knowledge agent (compare) produced answer (2483 chars)

🤖 Agent: knowledge_agent

**Understanding RISC vs CISC: A Student's Guide**

When comparing RISC (Reduced Instruction Set Computer) and CISC (Complex Instruction Set Computer), it's important to understand their design philosophies, advantages, disadvantages, and typical use cases. Here's a breakdown:

**Design Philosophy / Approach:**
- **RISC:**
  - Uses a small, highly optimized set of instructions.
  - Focuses on efficiency and speed by simplifying the instruction set.
- **CISC:**
  - Utilizes a large, complex set of instructions.
  - Aims to reduce the number of instructions per program by offering more complex operations.

**Advantages:**
- **RISC:**
  - Faster execution of instructions due to simpler hardware.
  - Easier to optimize and scale.
  - Lower power consumption.
- **CISC:**
  - Fewer instructions per program, which can reduce program size.
  - Complex instructions can perform multiple operations, potentially red

In [11]:
# Cell 5: Launch Gradio Chat UI (Interactive)
from app import create_ui, initialize as app_init

# Re-use the already loaded graph and memory
import app as app_module
app_module.graph = graph
app_module.memory = memory
app_module.config = config

demo = create_ui()
demo.launch(share=True, debug=True)  # share=True creates public URL

ModuleNotFoundError: No module named 'app'

In [18]:
# Cell 6: Alternative — Simple Chat Loop (no Gradio)
print('🤖 Agent is ready! Type "exit" to stop.\n')

while True:
    user_input = input('\nYou: ')

    if user_input.lower() in ['exit', 'quit', 'stop']:
        print('Goodbye!')
        break

    try:
        result = run_query(graph, user_input, memory)

        agent_names = {
            'rag_agent': '📚 Textbook',
            'math_agent': '🧮 Math',
            'knowledge_agent': '📖 Knowledge',
            'code_agent': '💻 Code',
            'direct': '💬 Chat',
        }
        agent_label = agent_names.get(result['agent_used'], result['agent_used'])

        print(f'\n[{agent_label} Agent]')
        print(f'Assistant: {result["response"]}')

        if result.get('tool_calls_log'):
            print(f'\n🔧 Tools: {", ".join(result["tool_calls_log"])}')
        print('-' * 50)
    except Exception as e:
        print(f'❌ Error: {e}')

🤖 Agent is ready! Type "exit" to stop.


You: what is BTB 
🔀 Supervisor routed to: RAG_AGENT
📚 RAG agent produced answer (1400 chars, 0 sources)

[📚 Textbook Agent]
Assistant: The Branch Target Buffer (BTB) is a high-speed memory component used in computer architecture, specifically in the context of branch prediction. Here's a breakdown of its function and relevance:

- The BTB stores predicted addresses for the next instruction after a branch instruction is encountered. This prediction is made before the instruction is fully decoded.
- It is indexed by the lower few bits of the branch instruction address, allowing for quick access to the predicted target address.
- The BTB contains information about whether the branch was predicted as taken (T) or not taken (NT).
- When the PC (Program Counter) of the fetched instruction matches one of the stored addresses in the BTB, it indicates a match with the predicted branch.
- If the match is correct, the fetched instruction can continue execu